# Ingest Orders Dataset file

1. read the file using spark dataframe reader API
- Define the Schema 
2. Add metadata columns
- source file 
- ingestion timestamp
3. write to the bronze delta table

In [0]:
%run ../01-common/01.bronze_helper

In [0]:
#Imports
from pyspark.sql.functions import col
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

In [0]:
source_file="/Volumes/olist_catalog/landing/files/olist_orders_dataset.csv"
table_name = "olist_catalog.bronze.orders"

In [0]:
orders_schema = StructType([
    StructField("order_id",StringType()),
    StructField("customer_id",StringType()),
    StructField("order_status",StringType()),
    StructField("order_purchase_timestamp",TimestampType()),
    StructField("order_approved_at",TimestampType()),
    StructField("order_delivered_carrier_date",TimestampType()),
    StructField("order_delivered_customer_date",TimestampType()),
    StructField("order_estimated_delivery_date",TimestampType())
])

In [0]:
orders_df = (
    spark.read
    .format("csv")
    .option("header","True")
    .schema(orders_schema)
    .load(source_file)
    )

In [0]:
orders_df_final=add_ingestion_data(orders_df)

In [0]:
display(orders_df_final)

In [0]:
(
    orders_df_final.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
select * from olist_catalog.bronze.orders